In [ ]:
!pip -q install ultralytics opencv-python pillow matplotlib tqdm

import os, json, random, math, shutil, time
from pathlib import Path

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

from ultralytics import YOLO


In [ ]:
WEIGHTS = "/content/weights.pt"

IN_DIR  = "/content/drive/MyDrive/pig-selected_frame_attribute_(5)/images_frame_attribute_(5)"
PREVIEW_DIR = "/content/preview_10"
OUT_IMG_DIR = None
# File COCO output
OUT_COCO = "/content/drive/MyDrive/pig-selected_frame_attribute_(5)/images_frame_attribute_(5)/annotate(5).coco.json"

CONF_THR   = 0.60       # 60%
IOU_THR    = 0.80       # 80% (NMS IoU)
IMGSZ      = 640
MAX_DET    = 300
AGNOSTIC_NMS = False

BOX_THICKNESS = 2
FONT_SCALE    = 0.6
OPACITY       = 0.75
SEED          = 1337

os.makedirs(PREVIEW_DIR, exist_ok=True)
#os.makedirs(OUT_IMG_DIR, exist_ok=True)

model = YOLO(WEIGHTS)

names = model.names if isinstance(model.names, list) else list(model.names.values())
if not names:
    names = ["pig"]
print(f"Classes: {names}")


In [ ]:
def ensure_uint8(img):
    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)
    return img

def draw_box_opacity(img_bgr, xyxy, label, color=(0, 255, 0), alpha=0.75):
    """
    xyxy: [x1, y1, x2, y2]
    """
    x1, y1, x2, y2 = map(int, xyxy)
    h, w = img_bgr.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w-1, x2), min(h-1, y2)

    overlay = img_bgr.copy()
    cv2.rectangle(overlay, (x1, y1), (x2, y2), color, -1)  # filled
    cv2.addWeighted(overlay, alpha, img_bgr, 1 - alpha, 0, dst=img_bgr)

    cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color, BOX_THICKNESS)
    (tw, th), bl = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, 2)
    y_text = max(th + 4, y1)
    cv2.rectangle(img_bgr, (x1, y_text - th - 4), (x1 + tw + 4, y_text + 4), color, -1)
    cv2.putText(img_bgr, label, (x1 + 2, y_text), cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, (0,0,0), 2, cv2.LINE_AA)

def coco_init(categories):
    return {
        "images": [],
        "annotations": [],
        "categories": [{"id": i+1, "name": c} for i, c in enumerate(categories)]
    }

def coco_add_image(coco, img_id, file_name, width, height):
    coco["images"].append({
        "id": img_id,
        "file_name": file_name,
        "width": width,
        "height": height
    })

def coco_add_ann(coco, ann_id, img_id, cat_id, xyxy):
    # COCO bbox = [x_min, y_min, width, height]
    x1, y1, x2, y2 = map(int, xyxy)
    w = max(0, x2 - x1)
    h = max(0, y2 - y1)
    coco["annotations"].append({
        "id": ann_id,
        "image_id": img_id,
        "category_id": cat_id,
        "bbox": [x1, y1, w, h],
        "area": int(w * h),
        "iscrowd": 0
    })


In [ ]:
exts = (".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff")
all_imgs = [p for p in sorted(Path(IN_DIR).glob("*")) if p.suffix.lower() in exts]

assert len(all_imgs) > 0

random.seed(SEED)
sample_imgs = random.sample(all_imgs, k=min(10, len(all_imgs)))
pass

for p in sample_imgs:
    results = model.predict(
        source=str(p),
        imgsz=IMGSZ,
        conf=CONF_THR,
        iou=IOU_THR,
        agnostic_nms=AGNOSTIC_NMS,
        max_det=MAX_DET,
        verbose=False
    )

    img_bgr = cv2.imread(str(p))
    img_bgr = ensure_uint8(img_bgr)

    r = results[0]
    if r.boxes is not None and len(r.boxes) > 0:
        xyxy = r.boxes.xyxy.cpu().numpy()
        conf = r.boxes.conf.cpu().numpy()
        cls  = r.boxes.cls.cpu().numpy().astype(int)

        for i in range(len(xyxy)):
            c = cls[i]
            score = float(conf[i])
            label = f"{names[c]} {score:.2f}"
            draw_box_opacity(img_bgr, xyxy[i], label, color=(0,255,0), alpha=OPACITY)

    out_path = str(Path(PREVIEW_DIR) / p.name)
    cv2.imwrite(out_path, img_bgr)

show_n = min(5, len(sample_imgs))
plt.figure(figsize=(16, 3 * show_n))
for i, p in enumerate(sample_imgs[:show_n], start=1):
    img = cv2.imread(str(Path(PREVIEW_DIR)/p.name))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(show_n, 1, i)
    plt.imshow(img)
    plt.axis("off")
    plt.title(p.name)
plt.tight_layout(); plt.show()


In [ ]:
from pathlib import Path
coco = coco_init(names)
ann_id = 1
img_id = 1

t0 = time.time()
total_boxes = 0

for p in tqdm(all_imgs, desc="Processing all images"):
    with Image.open(p) as im:
        width, height = im.size

    results = model.predict(
        source=str(p),
        imgsz=IMGSZ,
        conf=CONF_THR,
        iou=IOU_THR,
        agnostic_nms=AGNOSTIC_NMS,
        max_det=MAX_DET,
        verbose=False
    )
    r = results[0]

    coco_add_image(coco, img_id, p.name, width, height)

    if r.boxes is not None and len(r.boxes) > 0:
        xyxy = r.boxes.xyxy.cpu().numpy()
        cls  = r.boxes.cls.cpu().numpy().astype(int)

        for i in range(len(xyxy)):
            c = int(cls[i])
            coco_add_ann(coco, ann_id, img_id, c+1, xyxy[i])  # COCO bbox [x,y,w,h]
            ann_id += 1
            total_boxes += 1

    img_id += 1

with open(OUT_COCO, "w", encoding="utf-8") as f:
    json.dump(coco, f, ensure_ascii=False, indent=2)

pass
      f"COCO  {OUT_COCO}, Time: {time.time()-t0:.1f}s")
